In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully.")

All libraries imported successfully.


In [ ]:
np.random.seed(42)

n = 1000

study_hours = np.random.uniform(1, 10, n)
attendance = np.random.uniform(50, 100, n)
previous_marks = np.random.uniform(35, 95, n)
assignment_score = np.random.uniform(30, 100, n)
sleep_hours = np.random.uniform(4, 9, n)
participation = np.random.uniform(1, 10, n)

internet_access = np.random.choice(
    [0, 1],
    size=n,
    p=[0.15, 0.85]
)

# Create a performance score
performance_score = (
    study_hours * 5
    + attendance * 0.35
    + previous_marks * 0.40
    + assignment_score * 0.25
    + sleep_hours * 1.5
    + participation * 2
    + internet_access * 3
    + np.random.normal(0, 8, n)
)

# Generate target
final_result = (performance_score >= 75).astype(int)

data = pd.DataFrame({
    "Study_Hours": study_hours,
    "Attendance": attendance,
    "Previous_Marks": previous_marks,
    "Assignment_Score": assignment_score,
    "Sleep_Hours": sleep_hours,
    "Participation": participation,
    "Internet_Access": internet_access,
    "Final_Result": final_result
})

data.head()


In [ ]:
data.to_csv("student_performance.csv", index=False)

print("Dataset saved successfully.")

In [ ]:
data = pd.read_csv("student_performance.csv")

print(data.head())

In [ ]:
print("Shape of dataset:")
print(data.shape)

print("\nColumn names:")
print(data.columns)

print("\nDataset information:")
data.info()

In [ ]:
data.describe()

In [ ]:
print(data.isnull().sum())

In [ ]:
numeric_columns = data.select_dtypes(
    include=np.number
).columns

data[numeric_columns] = data[numeric_columns].fillna(
    data[numeric_columns].median()
)

print("Missing values handled.")

In [ ]:
print("Number of duplicate rows:",
      data.duplicated().sum())

In [ ]:
data = data.drop_duplicates()

print("Duplicates removed.")

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(
    x="Final_Result",
    data=data
)

plt.title("Distribution of Student Results")
plt.xlabel("Final Result")
plt.ylabel("Number of Students")

plt.xticks(
    [0, 1],
    ["Fail", "Pass"]
)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    x="Final_Result",
    y="Study_Hours",
    data=data
)

plt.title("Study Hours vs Final Result")
plt.xlabel("Final Result")
plt.ylabel("Study Hours")

plt.xticks(
    [0, 1],
    ["Fail", "Pass"]
)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    x="Final_Result",
    y="Attendance",
    data=data
)

plt.title("Attendance vs Final Result")
plt.xlabel("Final Result")
plt.ylabel("Attendance (%)")

plt.xticks(
    [0, 1],
    ["Fail", "Pass"]
)

plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

correlation = data.corr()

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
X = data.drop("Final_Result", axis=1)

y = data["Final_Result"]

print("Features:")
print(X.columns)

print("\nTarget:")
print(y.name)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")

In [ ]:
logistic_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

logistic_model.fit(
    X_train_scaled,
    y_train
)

print("Logistic Regression trained successfully.")

In [ ]:
decision_tree = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

decision_tree.fit(
    X_train,
    y_train
)

print("Decision Tree trained successfully.")

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(
    X_train,
    y_train
)

print("Random Forest trained successfully.")

In [ ]:
lr_prediction = logistic_model.predict(X_test_scaled)

lr_probability = logistic_model.predict_proba(
    X_test_scaled
)[:, 1]

dt_prediction = decision_tree.predict(X_test)

dt_probability = decision_tree.predict_proba(
    X_test
)[:, 1]

rf_prediction = random_forest.predict(X_test)

rf_probability = random_forest.predict_proba(
    X_test
)[:, 1]

In [ ]:
def evaluate_model(name, y_true, y_pred):
    
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    print(f"\n{name}")
    print("-" * 40)
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    
    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["Fail", "Pass"]
        )
    )

evaluate_model(
    "Logistic Regression",
    y_test,
    lr_prediction
)

evaluate_model(
    "Decision Tree",
    y_test,
    dt_prediction
)

evaluate_model(
    "Random Forest",
    y_test,
    rf_prediction
)

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    
    "Accuracy": [
        accuracy_score(y_test, lr_prediction),
        accuracy_score(y_test, dt_prediction),
        accuracy_score(y_test, rf_prediction)
    ],
    
    "Precision": [
        precision_score(y_test, lr_prediction),
        precision_score(y_test, dt_prediction),
        precision_score(y_test, rf_prediction)
    ],
    
    "Recall": [
        recall_score(y_test, lr_prediction),
        recall_score(y_test, dt_prediction),
        recall_score(y_test, rf_prediction)
    ],
    
    "F1_Score": [
        f1_score(y_test, lr_prediction),
        f1_score(y_test, dt_prediction),
        f1_score(y_test, rf_prediction)
    ]
})

results

In [ ]:
cm = confusion_matrix(
    y_test,
    rf_prediction
)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Fail", "Pass"],
    yticklabels=["Fail", "Pass"]
)

plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted Result")
plt.ylabel("Actual Result")

plt.show()

In [ ]:
lr_fpr, lr_tpr, _ = roc_curve(
    y_test,
    lr_probability
)

dt_fpr, dt_tpr, _ = roc_curve(
    y_test,
    dt_probability
)

rf_fpr, rf_tpr, _ = roc_curve(
    y_test,
    rf_probability
)

lr_auc = roc_auc_score(
    y_test,
    lr_probability
)

dt_auc = roc_auc_score(
    y_test,
    dt_probability
)

rf_auc = roc_auc_score(
    y_test,
    rf_probability
)

print("Logistic Regression AUC:", round(lr_auc, 4))
print("Decision Tree AUC:", round(dt_auc, 4))
print("Random Forest AUC:", round(rf_auc, 4))

plt.figure(figsize=(9, 6))

plt.plot(
    lr_fpr,
    lr_tpr,
    label=f"Logistic Regression (AUC = {lr_auc:.3f})"
)

plt.plot(
    dt_fpr,
    dt_tpr,
    label=f"Decision Tree (AUC = {dt_auc:.3f})"
)

plt.plot(
    rf_fpr,
    rf_tpr,
    label=f"Random Forest (AUC = {rf_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": random_forest.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=importance,
    x="Importance",
    y="Feature"
)

plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.show()

In [ ]:
new_student = pd.DataFrame({
    "Study_Hours": [6],
    "Attendance": [85],
    "Previous_Marks": [72],
    "Assignment_Score": [80],
    "Sleep_Hours": [7],
    "Participation": [8],
    "Internet_Access": [1]
})

prediction = random_forest.predict(new_student)

probability = random_forest.predict_proba(
    new_student
)[0][1]

if prediction[0] == 1:
    result = "PASS"
else:
    result = "FAIL"

print("Predicted Result:", result)
print(f"Probability of Passing: {probability:.2%}")

In [ ]:
def predict_student(
    study_hours,
    attendance,
    previous_marks,
    assignment_score,
    sleep_hours,
    participation,
    internet_access
):
    
    student = pd.DataFrame({
        "Study_Hours": [study_hours],
        "Attendance": [attendance],
        "Previous_Marks": [previous_marks],
        "Assignment_Score": [assignment_score],
        "Sleep_Hours": [sleep_hours],
        "Participation": [participation],
        "Internet_Access": [internet_access]
    })
    
    prediction = random_forest.predict(student)[0]
    
    probability = random_forest.predict_proba(
        student
    )[0][1]
    
    if prediction == 1:
        result = "PASS"
    else:
        result = "FAIL"
    
    return result, probability

result, probability = predict_student(
    study_hours=7,
    attendance=90,
    previous_marks=78,
    assignment_score=85,
    sleep_hours=7,
    participation=8,
    internet_access=1
)

print("Predicted Result:", result)
print(f"Pass Probability: {probability:.2%}")